# PatentLens - Model Training v2

**What's new vs. the first pass (`01_eda.ipynb`):**

1. **Scales past 3,000 patents.** Retrieval is done one query row at a time (via `src/patentlens/retrieval.py`), never as a full n x n similarity matrix, and the embedding models use a FAISS index. That's what let the original notebook's approach handle 3,000 patents but would have made it unusably slow/memory-heavy at 10-50k+ -- it doesn't need to change as the corpus grows.
2. **A patent-specific embedding model** (`AI-Growth-Lab/PatentSBERTa`, fine-tuned on patent claims) alongside general-purpose MiniLM, to see whether domain-specific pretraining actually helps.
3. **A hybrid model** combining BM25 (lexical) with the best embedding model via Reciprocal Rank Fusion -- often beats either alone.
4. **Richer metrics.** Recall@k alone doesn't show ranking quality or tell you how much to trust a small sample. This notebook adds Precision@k, MRR, and NDCG@k, all with bootstrap confidence intervals, plus a paired bootstrap significance test so "the best model" is a statistical claim, not just the top bar in a chart.
5. **Inventor/assignee metadata**, when sourced via BigQuery (`inventors`/`assignees` columns) -- surfaced in the Streamlit demo alongside each result.
6. **Saved artifacts.** Every fitted retriever gets persisted to `models/`, so the Streamlit app loads instantly instead of recomputing embeddings/indices on every run.

Set `FETCH_FROM_BIGQUERY = True` in the data-loading cell to pull a larger corpus than the
3,000-patent CSV checked into this repo -- see `src/patentlens/data_fetch.py` for the query
and a cost-estimation helper. Everything downstream works unchanged regardless of corpus size.


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from patentlens import cleaning, retrieval, evaluation

sns.set_style("whitegrid")
pd.set_option("display.max_colwidth", 80)


## 1. Load data

By default this loads the existing `data/raw/patents_g06n3_wide.csv` (~3,000 patents). To
scale up, flip `FETCH_FROM_BIGQUERY = True` below and set your GCP `PROJECT_ID` -- it will
dry-run the query first and print the estimated cost/bytes scanned before pulling anything.


In [ ]:
FETCH_FROM_BIGQUERY = False   # set True once you have a GCP project id to query with
PROJECT_ID = "your-gcp-project-id"
ROW_CAP = None                # None = pull everything BigQuery finds for the CPC prefix

RAW_CSV_FILENAME = "patents_g06n3_wide_100k.csv"  # swap back to patents_g06n3_wide.csv for the original 3k sample
RAW_CSV_PATH = PROJECT_ROOT / "data" / "raw" / RAW_CSV_FILENAME

if FETCH_FROM_BIGQUERY:
    from patentlens import data_fetch

    bytes_scanned, est_usd = data_fetch.estimate_query_cost(PROJECT_ID, row_cap=ROW_CAP)
    print(f"Dry run: {bytes_scanned / 1e9:.1f} GB scanned, ~${est_usd:.2f} estimated cost")
    print("Re-run this cell with FETCH_FROM_BIGQUERY left True only once you're happy with that estimate.")

    df = data_fetch.fetch_patents(PROJECT_ID, row_cap=ROW_CAP)
    df.to_csv(RAW_CSV_PATH, index=False)
    print(f"Saved {len(df)} rows to {RAW_CSV_PATH}")
else:
    df = pd.read_csv(RAW_CSV_PATH)

print("Loaded shape:", df.shape)
df.head(3)


## 2. Clean text and derive features

In [ ]:
df = cleaning.prepare_dataframe(df)

print(df['clean_word_count'].describe())
print("\nRows with fewer than 10 words after cleaning:", (df['clean_word_count'] < 10).sum())

n_with_inventors = (df['inventors'].apply(len) > 0).sum()
print(f"\nPatents with known inventors: {n_with_inventors} / {len(df)} "
      f"({n_with_inventors/len(df):.1%}) -- only populated when sourced via data_fetch.fetch_patents")

df[['publication_number', 'title', 'clean_word_count', 'primary_cpc_class']].head(3)


## 3. Build citation ground truth

Same idea as the original notebook: a patent's real citations, restricted to whichever cited
patents also fall inside our sampled corpus, become the evaluation ground truth for retrieval
quality. This coverage number should climb as the corpus grows, which makes every metric below
less noisy than it was on 3,000 patents.


In [ ]:
ground_truth, pub_to_idx = evaluation.build_ground_truth(df)

n_queries = len(ground_truth)
n_pairs = sum(len(v) for v in ground_truth.values())
print(f"Query patents with >=1 in-sample citation: {n_queries} / {len(df)} ({n_queries/len(df):.1%})")
print(f"Total usable citation pairs: {n_pairs}")


## 4. Build retrievers

Each retriever is fit once on `df['clean_text']` and exposes `.rank(query_idx, top_k)` for
evaluation and `.rank_text(free_text, top_k)` for live search in the Streamlit app.


In [ ]:
tfidf = retrieval.TfidfRetriever().fit(df['clean_text'])
print("TF-IDF matrix shape:", tfidf.matrix.shape)


In [ ]:
bm25 = retrieval.Bm25Retriever().fit(df['clean_text'])
print("BM25 fit on", len(bm25.tokenized_corpus), "documents")


In [ ]:
lsa = retrieval.LsaRetriever(tfidf, n_components=100).fit()
print("LSA matrix shape:", lsa.lsa_matrix.shape)
print("Variance explained:", round(lsa.svd.explained_variance_ratio_.sum(), 4))


In [ ]:
minilm = retrieval.EmbeddingRetriever(model_name="all-MiniLM-L6-v2", name="MiniLM").fit(
    df['clean_text'], batch_size=128
)
print("MiniLM embeddings shape:", minilm.embeddings.shape)


In [ ]:
patentsberta = retrieval.EmbeddingRetriever(
    model_name="AI-Growth-Lab/PatentSBERTa", name="PatentSBERTa"
).fit(df['clean_text'], batch_size=128)
print("PatentSBERTa embeddings shape:", patentsberta.embeddings.shape)


## 5. Hybrid model (BM25 + best embedding model, via Reciprocal Rank Fusion)

Lexical and semantic retrieval tend to make different mistakes, so fusing their rankings
(rather than picking one) is a common way to pick up both keyword-exact-match cases (BM25's
strength) and paraphrased/semantically-related cases (PatentSBERTa's strength).


In [ ]:
hybrid = retrieval.HybridRetriever([bm25, patentsberta], weights=[1.0, 1.0], name="Hybrid (BM25 + PatentSBERTa)")


## 6. Evaluate every model

In [ ]:
retrievers = {
    "TF-IDF": tfidf,
    "BM25": bm25,
    "LSA": lsa,
    "MiniLM": minilm,
    "PatentSBERTa": patentsberta,
    "Hybrid (BM25 + PatentSBERTa)": hybrid,
}

K_VALUES = (5, 10, 20)
MAX_EVAL_QUERIES = 3000  # TF-IDF/BM25 have no sublinear index -- ranking is O(corpus) per
                         # query, so evaluating tens of thousands of queries one at a time
                         # takes hours, not minutes. 3000 queries still gives much tighter
                         # confidence intervals than the original notebook's 127 ever had.

ground_truth_eval = evaluation.sample_ground_truth(ground_truth, max_queries=MAX_EVAL_QUERIES)
print(f"Evaluating on {len(ground_truth_eval)} of {len(ground_truth)} ground-truth queries")

all_summaries = {}
all_per_query = {}

for name, retriever in retrievers.items():
    rank_fn = lambda q, k, r=retriever: r.rank(q, top_k=k)[0]
    summary, per_query = evaluation.evaluate_retriever(rank_fn, ground_truth_eval, k_values=K_VALUES)
    all_summaries[name] = summary
    all_per_query[name] = per_query
    print(f"{name}: MRR={summary['MRR']['mean']:.4f}  Recall@10={summary['Recall@10']['mean']:.4f}  NDCG@10={summary['NDCG@10']['mean']:.4f}")


## 7. Results comparison

In [ ]:
summary_df = evaluation.summary_to_dataframe(all_summaries)
pivot = summary_df.pivot(index='model', columns='metric', values='mean')
metric_order = [f'{m}@{k}' for m in ['Recall', 'Precision', 'NDCG'] for k in K_VALUES] + ['MRR']
pivot = pivot[[c for c in metric_order if c in pivot.columns]].round(4)
pivot


In [ ]:
def plot_metric_with_ci(summary_df, metric_prefix, k_values, title):
    fig, ax = plt.subplots(figsize=(8, 4.5))
    models = list(summary_df['model'].unique())
    colors = sns.color_palette("Set2", len(models))
    x = np.arange(len(k_values))
    width = 0.8 / len(models)

    for i, model in enumerate(models):
        means, los, his = [], [], []
        for k in k_values:
            row = summary_df[(summary_df['model'] == model) & (summary_df['metric'] == f'{metric_prefix}@{k}')].iloc[0]
            means.append(row['mean'])
            los.append(row['mean'] - row['ci_low'])
            his.append(row['ci_high'] - row['mean'])
        ax.bar(x + i * width, means, width, label=model, color=colors[i],
               yerr=[los, his], capsize=3)

    ax.set_xticks(x + width * (len(models) - 1) / 2)
    ax.set_xticklabels([f'k={k}' for k in k_values])
    ax.set_ylabel(f"{metric_prefix}@k")
    ax.set_title(title, fontweight='bold')
    ax.legend(loc='upper left', fontsize=8, ncol=2)
    plt.tight_layout()
    return fig

fig1 = plot_metric_with_ci(summary_df, 'Recall', K_VALUES, "Recall@k with 95% bootstrap CI")
plt.savefig(PROJECT_ROOT / "outputs" / "recall_comparison.png", dpi=150)
plt.show()


In [ ]:
fig2 = plot_metric_with_ci(summary_df, 'NDCG', K_VALUES, "NDCG@k with 95% bootstrap CI (ranking quality)")
plt.savefig(PROJECT_ROOT / "outputs" / "ndcg_comparison.png", dpi=150)
plt.show()


In [ ]:
mrr_df = summary_df[summary_df['metric'] == 'MRR'].sort_values('mean', ascending=False)
fig, ax = plt.subplots(figsize=(7, 4))
colors = sns.color_palette("Set2", len(mrr_df))
bars = ax.bar(mrr_df['model'], mrr_df['mean'],
              yerr=[mrr_df['mean'] - mrr_df['ci_low'], mrr_df['ci_high'] - mrr_df['mean']],
              capsize=4, color=colors)
ax.set_ylabel("Mean Reciprocal Rank")
ax.set_title("MRR by model (higher = true citation ranked closer to #1)", fontweight='bold')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.savefig(PROJECT_ROOT / "outputs" / "mrr_comparison.png", dpi=150)
plt.show()


## 7b. Is the best model actually better, or just lucky?

A higher mean MRR isn't automatically a real difference -- with only ~127-150 usable
citation pairs, ranking noise alone can separate two models by a few points. A paired
bootstrap test resamples the *same* queries for both models and checks how often the
observed gap would vanish or reverse, which is a much stronger claim than "the bars don't
overlap."


In [ ]:
best_model_name_by_mrr = summary_df[summary_df['metric'] == 'MRR'].sort_values('mean', ascending=False).iloc[0]['model']
best_per_query_mrr = all_per_query[best_model_name_by_mrr]['MRR']

sig_rows = []
for name, per_query in all_per_query.items():
    if name == best_model_name_by_mrr:
        continue
    result = evaluation.paired_bootstrap_test(per_query['MRR'], best_per_query_mrr)
    sig_rows.append({
        'best_model': best_model_name_by_mrr,
        'compared_to': name,
        'mrr_mean_diff': round(result['mean_diff'], 4),
        'ci_low': round(result['ci_low'], 4),
        'ci_high': round(result['ci_high'], 4),
        'p_value': round(result['p_value'], 4),
        'significant_at_0.05': result['significant_at_0.05'],
    })

significance_df = pd.DataFrame(sig_rows).sort_values('p_value')
print(f"Best model by mean MRR: {best_model_name_by_mrr}\n")
significance_df


## 8. Qualitative sanity check

Numbers aside, spot-check that the top model's neighbors actually look related -- useful
both as a sanity check and as material for the live demo.


In [ ]:
best_model_name = best_model_name_by_mrr
best_retriever = retrievers[best_model_name]
print("Best model by MRR:", best_model_name)

test_idx = 0
idxs, scores = best_retriever.rank(test_idx, top_k=5)
print("\nQuery patent:", df.iloc[test_idx]['publication_number'], "-", df.iloc[test_idx]['title'])
print("\nTop 5 similar patents:")
pd.DataFrame({
    'publication_number': df.iloc[idxs]['publication_number'].values,
    'title': df.iloc[idxs]['title'].values,
    'score': np.round(scores, 4),
})


## 9. Save artifacts for the Streamlit app

Everything the app needs to run without recomputing: the cleaned dataframe, every fitted
retriever, and the metrics summary (for the in-app comparison tab).


In [ ]:
import joblib

MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(exist_ok=True)

df.to_parquet(MODELS_DIR / "patents.parquet")

tfidf.save(MODELS_DIR / "tfidf.joblib")
bm25.save(MODELS_DIR / "bm25.joblib")
lsa.save(MODELS_DIR / "lsa.joblib")
minilm.save(MODELS_DIR / "minilm")
patentsberta.save(MODELS_DIR / "patentsberta")

summary_df.to_csv(MODELS_DIR / "metrics_summary.csv", index=False)
pivot.to_csv(MODELS_DIR / "metrics_pivot.csv")
significance_df.to_csv(MODELS_DIR / "significance_tests.csv", index=False)

print("Saved artifacts to", MODELS_DIR)
for p in sorted(MODELS_DIR.rglob("*")):
    if p.is_file():
        print(" ", p.relative_to(MODELS_DIR))
